In [1]:
#!pip install pymupdf tiktoken sentence-transformers qdrant-client openai langchain -q
#!pip install -q langchain-text-splitters
#!pip install -q langchain langchain-text-splitters tiktoken

In [2]:
#!pip install -q \
#    ragas \
 #   datasets \
  #  langchain \
   # langchain-core \
    #langchain-community \
    #langchain-text-splitters \
    #langchain-groq

In [3]:
#!pip install -q pymupdf tiktoken sentence-transformers qdrant-client openai
#!pip install -q langchain langchain-core langchain-text-splitters langchain-groq
#!pip install -q "langchain-community<0.2.0"
#!pip install -q ragas datasets

In [4]:
!pip install -q pymupdf tiktoken sentence-transformers qdrant-client openai groq \
    langchain langchain-core langchain-text-splitters langchain-groq \
    ragas datasets

In [5]:
from google.colab import userdata
import os


# Qdrant
QDRANT_URL = userdata.get("QDRANT_URL")
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")

In [6]:
from google.colab import files
uploaded = files.upload()

Saving EmployeeHandbook.pdf to EmployeeHandbook (1).pdf
Saving Tennesse optionB.pdf to Tennesse optionB (1).pdf
Saving Tennessee Comprehensive Driver License Manual.pdf to Tennessee Comprehensive Driver License Manual (1).pdf


In [7]:
import fitz  # PyMuPDF

PDF_FILES = [
    "EmployeeHandbook.pdf",
    "Tennesse optionB.pdf",
    "Tennessee Comprehensive Driver License Manual.pdf",
]

def extract_pages(pdf_path):
    """Returns a list of dicts: one per page, with text + source metadata."""
    doc = fitz.open(pdf_path)
    pages = []
    for page_num, page in enumerate(doc, start=1):  # 1-indexed to match how humans read PDFs
        text = page.get_text()
        if text.strip():  # skip blank pages
            pages.append({
                "text": text,
                "source": pdf_path,
                "page": page_num
            })
    doc.close()
    return pages

all_pages = []
for pdf_path in PDF_FILES:
    pages = extract_pages(pdf_path)
    all_pages.extend(pages)
    print(f"{pdf_path}: {len(pages)} pages extracted")

print(f"\nTotal pages across all documents: {len(all_pages)}")

EmployeeHandbook.pdf: 48 pages extracted
Tennesse optionB.pdf: 14 pages extracted
Tennessee Comprehensive Driver License Manual.pdf: 133 pages extracted

Total pages across all documents: 195


In [8]:
print(all_pages[0]["source"], "- page", all_pages[0]["page"])
print(all_pages[0]["text"][:300])

EmployeeHandbook.pdf - page 1
Tennessee State 
Government 
Employee Handbook
Previous versions of the handbook may contain inaccurate information 
and should not be relied upon. For the most up to date information 
please reference the online version only.



In [9]:
import tiktoken
from langchain_text_splitters import RecursiveCharacterTextSplitter



encoding = tiktoken.get_encoding("cl100k_base")  # same encoding used by GPT-3.5/4 family

def token_length(text):
    return len(encoding.encode(text))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,        # target ~900 tokens per chunk
    chunk_overlap=150,     # ~150 token overlap between chunks
    length_function=token_length,
)

all_chunks = []
for page in all_pages:
    chunks = text_splitter.split_text(page["text"])
    for i, chunk_text in enumerate(chunks):
        all_chunks.append({
            "text": chunk_text,
            "source": page["source"],
            "page": page["page"],
            "chunk_id": f"{page['source']}_p{page['page']}_c{i}"
        })

print(f"Total chunks created: {len(all_chunks)}")

# quick sanity check: token size distribution
sizes = [token_length(c["text"]) for c in all_chunks]
print(f"Min: {min(sizes)}, Max: {max(sizes)}, Avg: {sum(sizes)/len(sizes):.0f}")

Total chunks created: 300
Min: 1, Max: 890, Avg: 576


In [10]:
MIN_CHUNK_TOKENS = 20  # discard anything below this — likely just headers/noise

before = len(all_chunks)
all_chunks = [c for c in all_chunks if token_length(c["text"]) >= MIN_CHUNK_TOKENS]
after = len(all_chunks)

print(f"Filtered {before - after} near-empty chunks, {after} remain")

sizes = [token_length(c["text"]) for c in all_chunks]
print(f"Min: {min(sizes)}, Max: {max(sizes)}, Avg: {sum(sizes)/len(sizes):.0f}")

Filtered 4 near-empty chunks, 296 remain
Min: 41, Max: 890, Avg: 584


In [11]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in all_chunks]
embeddings = embedding_model.encode(texts, batch_size=32, show_progress_bar=True)

print(f"Embedding shape: {embeddings.shape}")  # should be (296, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding shape: (296, 384)


In [12]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION_NAME = "mindscreen_rag_docs"  # or whatever name you prefer

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

print("Collection created:", client.get_collections())

/tmp/ipykernel_16201/3639068391.py:8: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Collection created: collections=[CollectionDescription(name='mindscreen_rag_docs')]


In [13]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "text": all_chunks[i]["text"],
            "source": all_chunks[i]["source"],
            "page": all_chunks[i]["page"],
            "chunk_id": all_chunks[i]["chunk_id"],
        }
    )
    for i in range(len(all_chunks))
]

client.upsert(collection_name=COLLECTION_NAME, points=points)

# confirm it landed
info = client.get_collection(COLLECTION_NAME)
print(f"Points in collection: {info.points_count}")

Points in collection: 296


In [14]:
def retrieve(query, top_k=5):
    query_vector = embedding_model.encode(query).tolist()
    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
    )
    return results.points  # note: .points, since query_points wraps results differently

test_query = "How many sick leave days do employees accrue per year?"
results = retrieve(test_query)

for r in results:
    print(f"Score: {r.score:.3f} | Source: {r.payload['source']} | Page: {r.payload['page']}")
    print(f"Text: {r.payload['text'][:200]}...")
    print("---")

Score: 0.724 | Source: Tennesse optionB.pdf | Page: 6
Text: 8, 2017. Amendments filed January 18, 2023; effective April 18, 2023. Amendments filed July 11, 2024; 
effective October 9, 2024. 
 
1120-06-.12 SICK LEAVE. Sick leave is accrued by an employee at the...
---
Score: 0.607 | Source: Tennesse optionB.pdf | Page: 6
Text: ATTENDANCE AND LEAVE 
 
CHAPTER 1120-06 
 
(Rule 1120-06-.11, continued) 
October, 2024 (Revised) 
6 
(3) 
Creditable Service. Any month, which was a part of a fifteen hundred sixty (1,560) hours or 
...
---
Score: 0.589 | Source: Tennesse optionB.pdf | Page: 9
Text: 58-2-101, if the leave of absence is at the request of the employee’s wing commander or the 
wing commander’s designated representative. In addition, leave without pay shall be granted 
to employees o...
---
Score: 0.588 | Source: Tennesse optionB.pdf | Page: 10
Text: of birth or; 
 
(b) 
The placement of a child with the employee for adoption and to bond with the newly 
placed child within one (1) ye

In [15]:
distractor_query = "What do I need to bring to register my vehicle for parking at my state office?"
results = retrieve(distractor_query)

for r in results:
    print(f"Score: {r.score:.3f} | Source: {r.payload['source']} | Page: {r.payload['page']}")
    print(f"Text: {r.payload['text'][:200]}...")
    print("---")

Score: 0.543 | Source: Tennessee Comprehensive Driver License Manual.pdf | Page: 81
Text: highway except for an emergency.
• Walking on the interstate is both illegal and dangerous.
Except for extreme emergency cases, remain in your
disabled vehicle until a State Trooper, other police offi...
---
Score: 0.464 | Source: Tennessee Comprehensive Driver License Manual.pdf | Page: 80
Text: 65
Parking
Drivers are responsible for making sure that their vehicles are
not hazards when parked.  Whenever you park, be sure it is in
a place that is far enough from any travel lane to avoid
interf...
---
Score: 0.439 | Source: Tennessee Comprehensive Driver License Manual.pdf | Page: 38
Text: 23
Also required:
Windows and windshields. Clear vision for the driver is 
required to the front, rear and both sides. It is unlawful to drive 
a motor vehicle with a windshield that is so cracked, or...
---
Score: 0.435 | Source: Tennessee Comprehensive Driver License Manual.pdf | Page: 81
Text: 66
Other parking

In [16]:
from google.colab import files
uploaded = files.upload()  # select the corrected golden_dataset.json

Saving golden_dataset.json to golden_dataset.json


In [17]:
import json

with open("golden_dataset.json") as f:
    golden = json.load(f)

def evaluate_retrieval(golden, top_k=5):
    results_log = []

    for item in golden:
        expected_sources = item["source_doc"]  # list, or None for trick questions

        if expected_sources is None:
            # trick question — nothing to check against retrieval, skip scoring here
            results_log.append({
                "id": item["id"],
                "difficulty": item["difficulty"],
                "question": item["question"],
                "hit": None,  # not applicable
            })
            continue

        retrieved = retrieve(item["question"], top_k=top_k)
        retrieved_sources = [r.payload["source"] for r in retrieved]

        # did ANY of the expected valid sources show up in the top_k?
        hit = any(src in retrieved_sources for src in expected_sources)

        results_log.append({
            "id": item["id"],
            "difficulty": item["difficulty"],
            "question": item["question"],
            "expected_sources": expected_sources,
            "retrieved_sources": retrieved_sources,
            "hit": hit,
        })

    return results_log

eval_results = evaluate_retrieval(golden, top_k=5)

# Overall precision (excluding trick questions, which aren't scored this way)
scored = [r for r in eval_results if r["hit"] is not None]
precision_at_5 = sum(r["hit"] for r in scored) / len(scored)

print(f"Precision@5 (source retrieval): {precision_at_5:.1%}  ({sum(r['hit'] for r in scored)}/{len(scored)})\n")

# Breakdown by difficulty
from collections import defaultdict
by_difficulty = defaultdict(list)
for r in scored:
    by_difficulty[r["difficulty"]].append(r["hit"])

for diff, hits in by_difficulty.items():
    print(f"  {diff:12s}: {sum(hits)}/{len(hits)} correct ({sum(hits)/len(hits):.0%})")

print("\nFailures:")
for r in scored:
    if not r["hit"]:
        print(f"  [{r['id']}] {r['question']}")
        print(f"      expected: {r['expected_sources']}")
        print(f"      got:      {r['retrieved_sources']}")

Precision@5 (source retrieval): 87.5%  (14/16)

  easy        : 6/7 correct (86%)
  medium      : 6/6 correct (100%)
  distractor  : 2/3 correct (67%)

Failures:
  [6] What is the maximum number of days a suspension without pay can last within a 12-month period?
      expected: ['EmployeeHandbook.pdf']
      got:      ['Tennesse optionB.pdf', 'Tennesse optionB.pdf', 'Tennesse optionB.pdf', 'Tennesse optionB.pdf', 'Tennessee Comprehensive Driver License Manual.pdf']
  [15] What do I need to bring to register my vehicle for parking at my state office?
      expected: ['EmployeeHandbook.pdf']
      got:      ['Tennessee Comprehensive Driver License Manual.pdf', 'Tennessee Comprehensive Driver License Manual.pdf', 'Tennessee Comprehensive Driver License Manual.pdf', 'Tennessee Comprehensive Driver License Manual.pdf', 'Tennessee Comprehensive Driver License Manual.pdf']


In [18]:
!pip install groq -q

In [19]:
from groq import Groq

groq_client = Groq(api_key=userdata.get("GROQ_API_KEY"))  # get a free key at console.groq.com

SYSTEM_PROMPT = """You are an enterprise knowledge assistant for Tennessee State Government employees.

Answer the user's question using ONLY the information in the provided context below. Do not use any outside knowledge.

If the context does not contain enough information to answer the question, respond exactly with:
"I don't have enough information to answer that question."

Always be precise with numbers, dates, and specific policy details — do not round or approximate figures from the context.

After your answer, list the sources you used in this format:
Sources: [document name, page X]"""

def format_context(retrieved_chunks):
    """Turn retrieved chunks into a labeled context block the LLM can cite from."""
    context_parts = []
    for chunk in retrieved_chunks:
        label = f"[{chunk.payload['source']}, page {chunk.payload['page']}]"
        context_parts.append(f"{label}\n{chunk.payload['text']}")
    return "\n\n---\n\n".join(context_parts)

def generate_answer(question, top_k=5):
    retrieved_chunks = retrieve(question, top_k=top_k)
    context = format_context(retrieved_chunks)

    user_message = f"""Context:
{context}

Question: {question}"""

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # fast, free-tier friendly
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ]
    )

    return {
        "answer": response.choices[0].message.content,
        "retrieved_chunks": retrieved_chunks,
    }

In [20]:
result = generate_answer("How many sick leave days do employees accrue per year?")
print(result["answer"])

Employees accrue twelve (12) days of sick leave per year.

Sources: [Tennesse optionB.pdf, page 6]


In [21]:
for q_id, question in [(6, "What is the maximum number of days a suspension without pay can last within a 12-month period?"),
                         (15, "What do I need to bring to register my vehicle for parking at my state office?")]:
    print(f"=== Question {q_id} ===")
    result = generate_answer(question)
    print(result["answer"])
    print()

=== Question 6 ===
I don't have enough information to answer that question.

Sources: None

=== Question 15 ===
I don't have enough information to answer that question.

Sources: None



In [22]:
import time

generation_log = []

for item in golden:
    print(f"Running Q{item['id']}...", end=" ")
    try:
        result = generate_answer(item["question"], top_k=5)
        generation_log.append({
            "id": item["id"],
            "difficulty": item["difficulty"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "expected_sources": item["source_doc"],
            "generated_answer": result["answer"],
            "retrieved_sources": [c.payload["source"] for c in result["retrieved_chunks"]],
            "retrieved_contexts": [c.payload["text"] for c in result["retrieved_chunks"]],
        })
        print("done")
    except Exception as e:
        print(f"FAILED: {e}")
        generation_log.append({
            "id": item["id"],
            "difficulty": item["difficulty"],
            "question": item["question"],
            "generated_answer": None,
            "error": str(e),
        })
    time.sleep(1)  # be gentle on Groq's free-tier rate limits

print(f"\nCompleted {len(generation_log)} questions")

# save to disk so you don't lose it if the runtime disconnects
with open("generation_log.json", "w") as f:
    json.dump(generation_log, f, indent=2)

Running Q1... done
Running Q2... done
Running Q3... done
Running Q4... done
Running Q5... done
Running Q6... done
Running Q7... done
Running Q8... done
Running Q9... done
Running Q10... done
Running Q11... done
Running Q12... done
Running Q13... done
Running Q14... done
Running Q15... done
Running Q16... done
Running Q17... done
Running Q18... done

Completed 18 questions


In [23]:
for entry in generation_log:
    print(f"[{entry['id']}] ({entry['difficulty']}) {entry['question']}")
    print(f"  → {entry['generated_answer'][:150]}...")
    print()

[1] (easy) How many days of annual leave do employees earn per year at minimum?
  → Employees earn at minimum twelve (12) annual leave days per year.

Sources: [Tennesse optionB.pdf, page 5]...

[2] (easy) How many sick leave days do employees accrue per year?
  → Employees accrue twelve (12) days of sick leave per year.

Sources: [Tennesse optionB.pdf, page 6]...

[3] (easy) How long is the unpaid lunch period for most employees?
  → The unpaid lunch period for most employees is one (1) hour.

Sources: [Tennesse optionB.pdf, page 4], [EmployeeHandbook.pdf, page 20]...

[4] (easy) What is the toll-free number to report fraud, waste, and abuse?
  → The toll-free number to report fraud, waste, and abuse is 1-800-232-5454.

Sources: [EmployeeHandbook.pdf, page 19]...

[5] (easy) How many hours of Emotional Wellbeing Solutions (EWS) work time can an employee use per year with supervisor approval?
  → 10 hours of work time per year with supervisor approval.

Sources: [EmployeeHandbook.pdf, 

In [25]:
!pip install -q --upgrade langchain-community

In [27]:
!pip install -q "ragas==0.1.9"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 2.5 MB/s eta 0:00:00


In [29]:
!pip install -q "langchain-community==0.2.16" "ragas==0.1.9"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 32.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.15.4 requires openai<3.0.0,>=2.0.0, but you have openai 1.109.1 which is incompatible.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-groq 1.1.3 requires langchain-core<2.0.0,>=1.4.0, but you ha

In [30]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision

# RAGAS needs ground_truth for context_precision — use your golden expected_answer
# skip trick questions (16, 17) since there's no retrieved context to evaluate against
scored_entries = [e for e in generation_log if e.get("generated_answer") and e["id"] not in (16, 17)]

ragas_data = {
    "question": [e["question"] for e in scored_entries],
    "answer": [e["generated_answer"] for e in scored_entries],
    "contexts": [e["retrieved_contexts"] for e in scored_entries],
    "ground_truth": [e["expected_answer"] for e in scored_entries],
}

ragas_dataset = Dataset.from_dict(ragas_data)

ragas_results = evaluate(
    ragas_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision],
)

print(ragas_results)
df = ragas_results.to_pandas()
df.to_csv("ragas_results.csv", index=False)
df[["question", "faithfulness", "answer_relevancy", "context_precision"]]

TypeError: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases